In [6]:
%pip install "numpy>=1.24" "torch>=2.0" "torchvision>=0.15"

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 13.3 MB/s  0:00:09 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 12.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [torch]  WARNING: The scripts torchfrtrace and torchrun are installed in '/Users/Taseen/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  NOTE: The current PATH contains path(s) starting with `~`, which may not be expanded by all applications.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchvision] 1/2 [torchvision]
Note: you may need to restart the kernel to use updated packages.


# Hardware Check and Directory Setup


In [3]:
import os
import torch

if not os.path.exists("train_log.py"):
    os.chdir("/Users/Taseen/Downloads/Semester 3/Comp Project/StudentKit/poison_dynamics_kit")
os.makedirs("runs", exist_ok=True)

print("GPU Available (CUDA):", torch.cuda.is_available())
print("Apple Silicon Available (MPS):", torch.backends.mps.is_available())

GPU Available (CUDA): False
Apple Silicon Available (MPS): True


# train_log.py with PreActResNet18 and VGG19BN

In [17]:
%%writefile train_log.py
import argparse, json, os, ssl
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import poison as P

# Disable SSL verification for dataset downloads on macOS
ssl._create_default_https_context = ssl._create_unverified_context

def load_dataset(name, root='./data', n_synth=4000):
    if name == 'synth':
        rng = np.random.default_rng(0)
        C, H, W, K = 1, 16, 16, 4
        templates = rng.standard_normal((K, C, H, W)).astype(np.float32)
        templates = templates / np.abs(templates).max()
        y = rng.integers(0, K, n_synth)
        X = templates[y] + 0.35 * rng.standard_normal((n_synth, C, H, W)).astype(np.float32)
        X = (X - X.min()) / (X.max() - X.min())
        ytr, Xtr = y[:int(.8 * n_synth)], X[:int(.8 * n_synth)]
        yte, Xte = y[int(.8 * n_synth):], X[int(.8 * n_synth):]
        return Xtr, ytr, Xte, yte, K
    from torchvision import datasets
    if name == 'mnist':
        tr = datasets.MNIST(root, train=True, download=True)
        te = datasets.MNIST(root, train=False, download=True)
        Xtr = tr.data.numpy()[:, None].astype(np.float32) / 255.0
        Xte = te.data.numpy()[:, None].astype(np.float32) / 255.0
        return Xtr, tr.targets.numpy(), Xte, te.targets.numpy(), 10
    if name == 'cifar10':
        tr = datasets.CIFAR10(root, train=True, download=True)
        te = datasets.CIFAR10(root, train=False, download=True)
        Xtr = tr.data.transpose(0, 3, 1, 2).astype(np.float32) / 255.0
        Xte = te.data.transpose(0, 3, 1, 2).astype(np.float32) / 255.0
        return Xtr, np.array(tr.targets), Xte, np.array(te.targets), 10
    raise ValueError(name)

class CNN(nn.Module):
    def __init__(self, c_in, n_cls, w=32):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(c_in, w, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(w, 2 * w, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(2), nn.Flatten())
        self.head = nn.Linear(2 * w * 4, n_cls)
    def features(self, x): return self.f(x)
    def forward(self, x): return self.head(self.f(x))

class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False)
    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut

class PreActResNet18(nn.Module):
    def __init__(self, c_in, n_cls, w=64):
        super().__init__()
        self.in_planes = w
        self.conv1 = nn.Conv2d(c_in, w, 3, stride=1, padding=1, bias=False)
        self.layer1 = self._make_layer(w, 2, 1)
        self.layer2 = self._make_layer(2 * w, 2, 2)
        self.layer3 = self._make_layer(4 * w, 2, 2)
        self.layer4 = self._make_layer(8 * w, 2, 2)
        self.bn = nn.BatchNorm2d(8 * w)
        self.head = nn.Linear(8 * w, n_cls)
    def _make_layer(self, planes, n_blocks, stride):
        strides = [stride] + [1] * (n_blocks - 1)
        layers = []
        for s in strides:
            layers.append(PreActBlock(self.in_planes, planes, s))
            self.in_planes = planes
        return nn.Sequential(*layers)
    def features(self, x):
        out = self.conv1(x)
        out = self.layer1(out); out = self.layer2(out)
        out = self.layer3(out); out = self.layer4(out)
        out = F.relu(self.bn(out))
        return F.adaptive_avg_pool2d(out, 1).flatten(1)
    def forward(self, x): return self.head(self.features(x))

_VGG19_CFG = [64,64,'M',128,128,'M',256,256,256,256,'M',512,512,512,512,'M',512,512,512,512,'M']
class VGG19BN(nn.Module):
    def __init__(self, c_in, n_cls):
        super().__init__()
        layers = []
        in_c = c_in
        for v in _VGG19_CFG:
            if v == 'M':
                layers.append(nn.MaxPool2d(2, 2))
            else:
                layers += [nn.Conv2d(in_c, v, 3, padding=1, bias=False),
                           nn.BatchNorm2d(v), nn.ReLU(inplace=True)]
                in_c = v
        self.f = nn.Sequential(*layers)
        self.head = nn.Linear(512, n_cls)
    def features(self, x): return self.f(x).flatten(1)
    def forward(self, x): return self.head(self.features(x))

@torch.no_grad()
def probe(model, X, y, device, bs=512):
    model.eval()
    L, C = [], []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i + bs]).to(device)
        yb = torch.from_numpy(y[i:i + bs]).long().to(device)
        logits = model(xb)
        L.append(F.cross_entropy(logits, yb, reduction='none').cpu().numpy())
        C.append(torch.softmax(logits, 1).gather(1, yb[:, None])[:, 0].cpu().numpy())
    model.train()
    return np.concatenate(L), np.concatenate(C)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--dataset', default='synth', choices=['synth', 'mnist', 'cifar10'])
    ap.add_argument('--attack', default='badnets', choices=list(P.ATTACKS) + ['none'])
    ap.add_argument('--rate', type=float, default=0.05)
    ap.add_argument('--target', type=int, default=0)
    ap.add_argument('--epochs', type=int, default=8)
    ap.add_argument('--bs', type=int, default=128)
    ap.add_argument('--lr', type=float, default=1e-3)
    ap.add_argument('--probe_every', type=int, default=20)
    ap.add_argument('--adaptive_shape', action='store_true')
    ap.add_argument('--adaptive', action='store_true')
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--out', default='dynamics.npz')
    ap.add_argument('--device', default='auto')
    ap.add_argument('--model', type=str, default='cnn', choices=['cnn', 'preactresnet18', 'vgg19bn'])
    a = ap.parse_args()

    dev = ('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu') if a.device == 'auto' else a.device
    torch.manual_seed(a.seed); np.random.seed(a.seed)

    Xtr, ytr, Xte, yte, K = load_dataset(a.dataset)
    if a.attack == 'none': m = np.zeros(len(Xtr), bool)
    elif a.attack == 'label_flip': Xtr, ytr, m = P.label_flip(Xtr, ytr, rate=a.rate, seed=a.seed, n_classes=K)
    else: Xtr, ytr, m = P.ATTACKS[a.attack](Xtr, ytr, target=a.target, rate=a.rate, seed=a.seed)
    print(f"device={dev}  dataset={a.dataset}  attack={a.attack}  poisoned {m.sum()}/{len(m)} ({100*m.mean():.1f}%)  classes={K}")

    if a.model == 'cnn': model = CNN(Xtr.shape[1], K).to(dev)
    elif a.model == 'preactresnet18': model = PreActResNet18(Xtr.shape[1], K).to(dev)
    elif a.model == 'vgg19bn': model = VGG19BN(Xtr.shape[1], K).to(dev)

    opt = torch.optim.Adam(model.parameters(), lr=a.lr)
    g = torch.Generator().manual_seed(a.seed)

    L_hist, C_hist, steps = [], [], []
    step = 0
    w_poison = 1.0
    m_t = torch.from_numpy(m.astype(np.float32)).to(dev)
    w_i = np.ones(len(Xtr), np.float32)
    twin = np.zeros(len(Xtr), int)
    if a.adaptive_shape and m.sum() > 0:
        twin[m] = np.random.default_rng(a.seed).choice(np.where(~m)[0], m.sum(), replace=True)
    w_t = torch.from_numpy(w_i).to(dev)
    l0, c0 = probe(model, Xtr, ytr, dev)
    L_hist.append(l0); C_hist.append(c0); steps.append(0)

    for ep in range(a.epochs):
        perm = torch.randperm(len(Xtr), generator=g).numpy()
        for i in range(0, len(perm) - a.bs + 1, a.bs):
            b = perm[i:i + a.bs]
            xb, yb = torch.from_numpy(Xtr[b]).to(dev), torch.from_numpy(ytr[b]).long().to(dev)
            opt.zero_grad()
            per = F.cross_entropy(model(xb), yb, reduction='none')
            if a.adaptive_shape: (per * w_t[b]).mean().backward()
            elif a.adaptive:
                wb = torch.where(m_t[b] > 0.5, torch.tensor(w_poison, device=dev), torch.tensor(1.0, device=dev))
                (per * wb).mean().backward()
            else: per.mean().backward()
            opt.step()
            step += 1
            if step % a.probe_every == 0:
                l, c = probe(model, Xtr, ytr, dev)
                L_hist.append(l); C_hist.append(c); steps.append(step)
                if a.adaptive and m.sum() > 0:
                    w_poison = float(np.clip(w_poison + 2.0 * (l[m].mean() - l[~m].mean()), 0.02, 2.0))
                if a.adaptive_shape and m.sum() > 0:
                    w_i[m] = np.clip(w_i[m] + 3.0 * (l[m] - l[twin[m]]), 0.01, 3.0)
                    w_t = torch.from_numpy(w_i).to(dev)
        print(f"  epoch {ep+1}/{a.epochs}  checkpoints so far {len(steps)}")

    with torch.no_grad():
        model.eval()
        pred = model(torch.from_numpy(Xte).to(dev)).argmax(1).cpu().numpy()
        clean_acc = float((pred == yte).mean())
        asr = float('nan')
        if a.attack in P.TRIGGER_ATTACKS:
            Xt, yt = P.trigger_test_set(Xte, yte, target=a.target, kind=a.attack, seed=a.seed)
            asr = float((model(torch.from_numpy(Xt).to(dev)).argmax(1).cpu().numpy() == yt).mean())

    os.makedirs(os.path.dirname(a.out), exist_ok=True)
    np.savez_compressed(a.out, loss=np.stack(L_hist, 1), conf=np.stack(C_hist, 1),
                        y=ytr, is_poisoned=m, steps=np.array(steps), adaptive=bool(a.adaptive),
                        config=json.dumps(vars(a)), clean_acc=clean_acc, asr=asr)
    print(f"wrote {a.out}")

if __name__ == '__main__': main()

Overwriting train_log.py


# detect.py patch

In [6]:
%%writefile detect.py
"""
detect.py -- detectors for poisoned / backdoored training samples.
(patched: matched_benign_null no longer re-sorts the full array per draw)
"""
import numpy as np

def zscore(x, axis=0, eps=1e-9):
    x = np.asarray(x, float)
    return (x - x.mean(axis, keepdims=True)) / (x.std(axis, keepdims=True) + eps)

def _pairwise_corr(M):
    Z = zscore(M, axis=1)
    n = len(Z)
    if n < 2:
        return 0.0
    C = (Z @ Z.T) / Z.shape[1]
    iu = np.triu_indices(n, 1)
    return float(np.nanmean(C[iu]))

def baseline_early_loss(loss, frac=0.15):
    k = max(1, int(frac * loss.shape[1]))
    return -loss[:, :k].mean(1)

def baseline_loss_auc(loss):
    return -loss.mean(1)

def baseline_loss_shape(loss, n_comp=3):
    Z = zscore(loss, axis=1)
    Z = Z - Z.mean(0, keepdims=True)
    U, S, Vt = np.linalg.svd(Z, full_matrices=False)
    P = U[:, :n_comp] * S[:n_comp]
    med = np.median(P, axis=0)
    return np.linalg.norm(P - med, axis=1)

def spectral_dynamics(loss, conf, k=2):
    dL = np.diff(loss, axis=1)
    dC = np.diff(conf, axis=1)
    F = np.hstack([dL, dC])
    F = F - F.mean(0, keepdims=True)
    F = F / (F.std(0, keepdims=True) + 1e-9)
    U, S, _ = np.linalg.svd(F, full_matrices=False)
    return np.linalg.norm(U[:, :k] * S[:k], axis=1)

def group_coherence(loss, conf, idx):
    dL = np.diff(loss[idx], axis=1)
    dC = np.diff(conf[idx], axis=1)
    return 0.5 * (_pairwise_corr(dL) + _pairwise_corr(dC))

def matched_benign_null(loss, conf, idx, rng=None, n_draw=200, pool=40):
    rng = np.random.default_rng(0) if rng is None else rng
    obs = group_coherence(loss, conf, idx)
    difficulty = loss.mean(1)
    order = np.argsort(difficulty)
    sorted_diff = difficulty[order]
    target = np.sort(difficulty[idx])
    positions = np.searchsorted(sorted_diff, target)

    n = len(difficulty)
    nulls = []
    for _ in range(n_draw):
        pick = []
        for pos in positions:
            lo = max(0, min(pos - pool // 2, n - pool))
            cand = order[lo:lo + pool]
            pick.append(rng.choice(cand))
        nulls.append(group_coherence(loss, conf, np.array(sorted(set(pick)))))
    nulls = np.array(nulls)
    z = (obs - nulls.mean()) / (nulls.std() + 1e-9)
    return obs, float(nulls.mean()), float(np.percentile(nulls, 95)), float(z)

DETECTORS = {
    'early_loss': lambda L, C: baseline_early_loss(L),
    'loss_auc': lambda L, C: baseline_loss_auc(L),
    'loss_shape': lambda L, C: baseline_loss_shape(L),
    'spectral_dyn': lambda L, C: spectral_dynamics(L, C),
}

Overwriting detect.py


# overwrite result output

In [8]:
%%writefile evaluate.py
import argparse, json
import numpy as np
from detect import DETECTORS, group_coherence, matched_benign_null

def auc(score, y):
    s = np.asarray(score, float)
    pos, neg = s[y == 1], s[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float('nan')
    order = np.argsort(s)
    ranks = np.empty(len(s)); ranks[order] = np.arange(1, len(s) + 1)
    return (ranks[y == 1].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))

def at_budget(score, y, budget):
    k = max(1, int(budget * len(y)))
    top = np.argsort(-np.asarray(score))[:k]
    caught = y[top].sum()
    return dict(k=k, recall=caught / max(y.sum(), 1), precision=caught / k,
                clean_removed=(k - caught) / max((y == 0).sum(), 1), idx=top)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('files', nargs='+')
    ap.add_argument('--budget', type=float, default=0.10)
    a = ap.parse_args()

    for f in a.files:
        z = np.load(f, allow_pickle=True)
        L, C, y = z['loss'], z['conf'], z['is_poisoned'].astype(int)
        cfg = json.loads(str(z['config']))
        print(f"\n{'='*74}")
        # Removed the file name {f} from this print statement
        print(f"attack={cfg['attack']} rate={cfg['rate']}  "
              f"poisoned {y.sum()}/{len(y)}  checkpoints {L.shape[1]}")
        print(f"clean test acc {float(z['clean_acc']):.3f}   attack success rate "
              f"{float(z['asr']):.3f}")
        if y.sum() == 0:
            print("  no poison in this run (attack=none): use it as a false-alarm test.")
        print(f"{'='*74}")
        print(f"  {'detector':<14}{'AUC':>7}{'recall':>9}{'precision':>11}"
              f"{'clean lost':>12}   at budget {a.budget:.0%}")
        
        best = None
        for name, fn in DETECTORS.items():
            s = fn(L, C)
            r = at_budget(s, y, a.budget)
            print(f"  {name:<14}{auc(s, y):>7.3f}{r['recall']:>9.2f}"
                  f"{r['precision']:>11.2f}{r['clean_removed']:>12.2%}")
            if name == 'spectral_dyn':
                best = r

        if best is not None and y.sum() > 0:
            obs, nm, n95, zz = matched_benign_null(L, C, best['idx'],
                                                   rng=np.random.default_rng(0), n_draw=60)
            print(f"\n  CONFIRMATION on the flagged set (shared-trigger detector):")
            print(f"    move-together score {obs:+.3f}   equally-easy benign groups "
                  f"{nm:+.3f} (95th pct {n95:+.3f})   z = {zz:+.1f}")
            verdict = ("CONFIRMED: moves together far more than equally-easy benign data."
                       if zz > 3 else
                       "NOT CONFIRMED: no more coherent than equally-easy benign data.")
            print(f"    {verdict}")
            if best['precision'] < 0.25:
                print(f"    [!] precision is only {best['precision']:.2f}, so this number "
                      f"describes whatever got flagged, NOT the poison. Read precision first.")

if __name__ == '__main__':
    main()

Overwriting evaluate.py


# CNN baseline run


In [9]:
!python3 train_log.py --dataset cifar10 --attack none --rate 0.05 --epochs 25 --probe_every 40 --out runs/dyn_none.npz
!python3 train_log.py --dataset cifar10 --attack badnets --rate 0.05 --epochs 25 --probe_every 40 --out runs/dyn_badnets.npz
!python3 train_log.py --dataset cifar10 --attack badnets --rate 0.05 --adaptive_shape --epochs 25 --probe_every 40 --out runs/dyn_adaptive_shape.npz
!python3 evaluate.py runs/dyn_none.npz runs/dyn_badnets.npz runs/dyn_adaptive_shape.npz --budget 0.10 | tee runs/results_cifar50k_v2.txt

Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/urllib/request.py", line 1344, in do_open
    h.request(req.get_method(), req.selector, req.data, headers,
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1336, in request
    self._send_request(method, url, body, headers, encode_chunked)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1382, in _send_request
    self.endheaders(body, encode_chunked=encode_chunked)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1331, in endheaders
    self._send_output(message_body, encode_chunked=encode_chunked)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1091, in _send_output
    self.send(msg)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1035, in sen

Early Checkpoints Analysis Scriptb

In [10]:
%%writefile check_early_checkpoints.py
import numpy as np
from detect import spectral_dynamics
from evaluate import at_budget, auc

for f in ["runs/dyn_badnets.npz", "runs/dyn_adaptive_shape.npz"]:
    z = np.load(f, allow_pickle=True)
    L, C, y = z['loss'], z['conf'], z['is_poisoned'].astype(int)
    n_ckpt = L.shape[1]
    print(f"\n=== {f} ({n_ckpt} checkpoints total) ===")
    for frac in [0.1, 0.25, 0.5, 1.0]:
        k = max(2, int(frac * n_ckpt))
        s = spectral_dynamics(L[:, :k], C[:, :k])
        r = at_budget(s, y, 0.10)
        print(f"  first {frac:.0%} of checkpoints ({k:>4} ckpts): "
              f"AUC={auc(s, y):.3f}  precision={r['precision']:.3f}  recall={r['recall']:.2f}")

Writing check_early_checkpoints.py


In [11]:
!python3 check_early_checkpoints.py


=== runs/dyn_badnets.npz (469 checkpoints total) ===
  first 10% of checkpoints (  46 ckpts): AUC=0.542  precision=0.028  recall=0.06
  first 25% of checkpoints ( 117 ckpts): AUC=0.555  precision=0.053  recall=0.11
  first 50% of checkpoints ( 234 ckpts): AUC=0.532  precision=0.026  recall=0.05
  first 100% of checkpoints ( 469 ckpts): AUC=0.495  precision=0.021  recall=0.04

=== runs/dyn_adaptive_shape.npz (19 checkpoints total) ===
  first 10% of checkpoints (   2 ckpts): AUC=0.988  precision=0.475  recall=0.95
  first 25% of checkpoints (   4 ckpts): AUC=0.967  precision=0.403  recall=0.81
  first 50% of checkpoints (   9 ckpts): AUC=0.886  precision=0.194  recall=0.39
  first 100% of checkpoints (  19 ckpts): AUC=0.974  precision=0.438  recall=0.88


10% Poison Rate Attack Matrix

In [12]:
!python3 train_log.py --dataset cifar10 --attack blended --rate 0.10 --epochs 25 --probe_every 40 --out runs/dyn_blended.npz
!python3 train_log.py --dataset cifar10 --attack warp --rate 0.10 --epochs 25 --probe_every 40 --out runs/dyn_warp.npz
!python3 train_log.py --dataset cifar10 --attack frequency --rate 0.10 --epochs 25 --probe_every 40 --out runs/dyn_frequency.npz
!python3 train_log.py --dataset cifar10 --attack clean_label --rate 0.10 --epochs 25 --probe_every 40 --out runs/dyn_clean_label.npz
!python3 train_log.py --dataset cifar10 --attack narcissus --rate 0.10 --epochs 25 --probe_every 40 --out runs/dyn_narcissus.npz
!python3 train_log.py --dataset cifar10 --attack label_flip --rate 0.10 --epochs 25 --probe_every 40 --out runs/dyn_label_flip.npz
!python3 train_log.py --dataset cifar10 --attack sample_specific --rate 0.10 --epochs 25 --probe_every 40 --out runs/dyn_sample_specific.npz
!python3 train_log.py --dataset cifar10 --attack badnets --rate 0.10 --adaptive --epochs 25 --probe_every 40 --out runs/dyn_adaptive_level.npz

!python3 evaluate.py runs/dyn_*.npz --budget 0.10 | tee runs/results_full_matrix.txt

Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/urllib/request.py", line 1344, in do_open
    h.request(req.get_method(), req.selector, req.data, headers,
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1336, in request
    self._send_request(method, url, body, headers, encode_chunked)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1382, in _send_request
    self.endheaders(body, encode_chunked=encode_chunked)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1331, in endheaders
    self._send_output(message_body, encode_chunked=encode_chunked)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1091, in _send_output
    self.send(msg)
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/http/client.py", line 1035, in sen

# Ground Truth Check Script

In [13]:
%%writefile gate1_ground_truth_check.py
import numpy as np, glob
from detect import matched_benign_null

for f in sorted(glob.glob("runs/dyn_*.npz")):
    z = np.load(f, allow_pickle=True)
    L, C, y = z['loss'], z['conf'], z['is_poisoned'].astype(int)
    if y.sum() == 0:
        print(f"{f}: no poison, skipped"); continue
    true_idx = np.where(y == 1)[0]
    obs, nm, n95, zz = matched_benign_null(L, C, true_idx, rng=np.random.default_rng(0), n_draw=30)
    print(f"{f:35s}  n_poison={len(true_idx):5d}  ground-truth z={zz:8.1f}")

Writing gate1_ground_truth_check.py


In [14]:
!python3 gate1_ground_truth_check.py

runs/dyn_adaptive_level.npz          n_poison=  160  ground-truth z=    33.5
runs/dyn_adaptive_shape.npz          n_poison=  160  ground-truth z=     3.2
runs/dyn_badnets.npz                 n_poison= 3000  ground-truth z=   722.2
runs/dyn_blended.npz                 n_poison= 3000  ground-truth z=   776.6
runs/dyn_clean_label.npz             n_poison=  160  ground-truth z=     1.2
runs/dyn_frequency.npz               n_poison=  160  ground-truth z=     5.0
runs/dyn_label_flip.npz              n_poison=  160  ground-truth z=    -0.5
runs/dyn_narcissus.npz               n_poison=  160  ground-truth z=    14.1
runs/dyn_none.npz: no poison, skipped
runs/dyn_sample_specific.npz         n_poison=  160  ground-truth z=     2.5
runs/dyn_warp.npz                    n_poison=  160  ground-truth z=     5.8


# PreActResNet18 & VGG19BN

In [18]:
!python3 train_log.py --dataset cifar10 --attack none --rate 0.05 --model preactresnet18 --epochs 25 --probe_every 40 --out runs/dyn_none_preact.npz
!python3 train_log.py --dataset cifar10 --attack badnets --rate 0.05 --model preactresnet18 --epochs 25 --probe_every 40 --out runs/dyn_badnets_preact.npz
!python3 train_log.py --dataset cifar10 --attack badnets --rate 0.05 --adaptive_shape --model preactresnet18 --epochs 25 --probe_every 40 --out runs/dyn_adaptive_shape_preact.npz

!python3 train_log.py --dataset cifar10 --attack none --rate 0.05 --model vgg19bn --epochs 25 --probe_every 40 --out runs/dyn_none_vgg.npz
!python3 train_log.py --dataset cifar10 --attack badnets --rate 0.05 --model vgg19bn --epochs 25 --probe_every 40 --out runs/dyn_badnets_vgg.npz
!python3 train_log.py --dataset cifar10 --attack badnets --rate 0.05 --adaptive_shape --model vgg19bn --epochs 25 --probe_every 40 --out runs/dyn_adaptive_shape_vgg.npz

!python3 evaluate.py runs/dyn_*_preact.npz runs/dyn_*_vgg.npz --budget 0.10 | tee runs/results_advanced_architectures.txt

100.0%
device=mps  dataset=cifar10  attack=none  poisoned 0/50000 (0.0%)  classes=10
  epoch 1/25  checkpoints so far 10
  epoch 2/25  checkpoints so far 20
  epoch 3/25  checkpoints so far 30
  epoch 4/25  checkpoints so far 40
  epoch 5/25  checkpoints so far 49
  epoch 6/25  checkpoints so far 59
  epoch 7/25  checkpoints so far 69
  epoch 8/25  checkpoints so far 79
  epoch 9/25  checkpoints so far 88
  epoch 10/25  checkpoints so far 98
  epoch 11/25  checkpoints so far 108
  epoch 12/25  checkpoints so far 118
  epoch 13/25  checkpoints so far 127
  epoch 14/25  checkpoints so far 137
  epoch 15/25  checkpoints so far 147
  epoch 16/25  checkpoints so far 157
  epoch 17/25  checkpoints so far 166
  epoch 18/25  checkpoints so far 176
  epoch 19/25  checkpoints so far 186
  epoch 20/25  checkpoints so far 196
  epoch 21/25  checkpoints so far 205
  epoch 22/25  checkpoints so far 215
  epoch 23/25  checkpoints so far 225
  epoch 24/25  checkpoints so far 235
  epoch 25/25  checkpo

In [6]:
!python3 signed_pc_scan.py "runs/dyn_*_preact.npz" "runs/dyn_*_vgg.npz"


=== runs/dyn_adaptive_shape_preact.npz  ckpts=244  poison=2500  acc=0.842 ===
spectral_dyn (top-2 norm)      AUC=0.162  precision@5%=0.003  (chance 0.050)
best single PCs (ORACLE):      PC12:0.29  PC19:0.70  PC29:0.69  PC22:0.33
unsupervised max-kurtosis PC40 AUC=0.629  precision@5%=0.005

=== runs/dyn_adaptive_shape_vgg.npz  ckpts=244  poison=2500  acc=0.832 ===
spectral_dyn (top-2 norm)      AUC=0.125  precision@5%=0.000  (chance 0.050)
best single PCs (ORACLE):      PC16:0.71  PC21:0.30  PC5:0.70  PC24:0.69
unsupervised max-kurtosis PC26 AUC=0.579  precision@5%=0.016

=== runs/dyn_badnets_preact.npz  ckpts=244  poison=2500  acc=0.848 ===
spectral_dyn (top-2 norm)      AUC=0.122  precision@5%=0.008  (chance 0.050)
best single PCs (ORACLE):      PC10:0.67  PC22:0.34  PC14:0.61  PC19:0.60
unsupervised max-kurtosis PC16 AUC=0.448  precision@5%=0.030

=== runs/dyn_badnets_vgg.npz  ckpts=244  poison=2500  acc=0.860 ===
spectral_dyn (top-2 norm)      AUC=0.093  precision@5%=0.007  (chance

In [9]:
!python3 positive_control.py "runs/dyn_none_preact.npz" "runs/dyn_none_vgg.npz"

POSITIVE CONTROL: can the real spectral_dynamics detector find a PLANTED coherent signal?

=== runs/dyn_none_preact.npz   n=50000  planted=2500 (5%)  ckpts=244 ===
  no signal (random labels):   AUC=0.494   <- should be ~0.5
  planted strength  0.5:        AUC=0.998  precision@5%=0.937  <- DETECTOR WORKS
  planted strength  1.0:        AUC=1.000  precision@5%=1.000  <- DETECTOR WORKS
  planted strength  2.0:        AUC=1.000  precision@5%=1.000  <- DETECTOR WORKS
  planted strength  4.0:        AUC=1.000  precision@5%=1.000  <- DETECTOR WORKS

=== runs/dyn_none_vgg.npz   n=50000  planted=2500 (5%)  ckpts=244 ===
  no signal (random labels):   AUC=0.496   <- should be ~0.5
  planted strength  0.5:        AUC=0.931  precision@5%=0.318  <- DETECTOR WORKS
  planted strength  1.0:        AUC=1.000  precision@5%=1.000  <- DETECTOR WORKS
  planted strength  2.0:        AUC=1.000  precision@5%=1.000  <- DETECTOR WORKS
  planted strength  4.0:        AUC=1.000  precision@5%=1.000  <- DETECTOR W

In [11]:
!python3 -c "import numpy as np; z=np.load('runs/dyn_badnets_preact.npz',allow_pickle=True); print('keys:',list(z.keys())); print('loss shape:',z['loss'].shape); print('poison count:',int(z['is_poisoned'].sum()))"

keys: ['loss', 'conf', 'y', 'is_poisoned', 'steps', 'adaptive', 'config', 'clean_acc', 'asr']
loss shape: (50000, 244)
poison count: 2500


# Baseline Compare

In [12]:
!python3 baseline_compare.py "runs/dyn_*_preact.npz" "runs/dyn_*_vgg.npz"

TABLE 1 -- detector comparison (budget = 10%, higher AUC/recall/precision = better)

--- dyn_adaptive_shape_preact.npz   clean_acc=0.842  ASR=0.972  poison=2500/50000 ---
    detector                                AUC   recall  precision
    early_loss        (ABL-style)         0.635    0.021      0.010
    loss_auc          (BaDLoss-style)     0.706    0.028      0.014
    loss_shape        (Telltale-style)    0.687    0.012      0.006
    spectral_sig_dyn  (Tran 2018-style)   0.144    0.004      0.002
    spectral_dyn      (OURS)              0.162    0.009      0.005

--- dyn_adaptive_shape_vgg.npz   clean_acc=0.832  ASR=0.973  poison=2500/50000 ---
    detector                                AUC   recall  precision
    early_loss        (ABL-style)         0.751    0.144      0.072
    loss_auc          (BaDLoss-style)     0.824    0.227      0.114
    loss_shape        (Telltale-style)    0.769    0.033      0.017
    spectral_sig_dyn  (Tran 2018-style)   0.248    0.008      0.0

# Train adaptive_level, then re-run the comparison

In [ ]:
!python3 train_log.py --dataset cifar10 --attack badnets --rate 0.05 --adaptive --model preactresnet18 --epochs 25 --probe_every 40 --out runs/dyn_adaptive_level_preact.npz

device=mps  dataset=cifar10  attack=badnets  poisoned 2500/50000 (5.0%)  classes=10


###### [-'p;GTSRB GTSRB

In [ ]:
!python3 train_log.py --dataset gtsrb --attack none --rate 0.05 --target 0 --model preactresnet18 --epochs 25 --probe_every 40 --out runs/dyn_none_gtsrb.npz

!python3 train_log.py --dataset gtsrb --attack badnets --rate 0.05 --target 0 --model preactresnet18 --epochs 25 --probe_every 40 --out runs/dyn_badnets_gtsrb.npz

!python3 train_log.py --dataset gtsrb --attack badnets --rate 0.05 --target 0 --adaptive_shape --model preactresnet18 --epochs 25 --probe_every 40 --out runs/dyn_adaptive_shape_gtsrb.npz

100%|███████████████████████████████████████| 187M/187M [01:03<00:00, 2.94MB/s]
100%|█████████████████████████████████████| 89.0M/89.0M [00:27<00:00, 3.25MB/s]
100%|█████████████████████████████████████| 99.6k/99.6k [00:01<00:00, 84.3kB/s]
device=mps  dataset=gtsrb  attack=none  poisoned 0/26640 (0.0%)  classes=43
  epoch 1/25  checkpoints so far 6
  epoch 2/25  checkpoints so far 11
  epoch 3/25  checkpoints so far 16
  epoch 4/25  checkpoints so far 21
  epoch 5/25  checkpoints so far 27
  epoch 6/25  checkpoints so far 32
  epoch 7/25  checkpoints so far 37
  epoch 8/25  checkpoints so far 42
  epoch 9/25  checkpoints so far 47
  epoch 10/25  checkpoints so far 53
